# Безопасность и SSO

## Общая схема

Приложение остается OIDC-клиентом и resource server

Оно не знает деталей AD, FreeIPA, LDAP или внешнего провайдера

Схема:

1. Пользователь открывает публичный или внутренний адрес
2. Spring Security отправляет пользователя в Keycloak
3. Keycloak выбирает нужный способ входа
4. После входа приложение получает JWT
5. Роли берутся из claims Keycloak
6. Доменные права уточняются через `patient_accounts` и `staff_accounts`

## Клиенты Keycloak

Используются два клиента:

- `medical-appointment-public`
- `medical-appointment-internal`

Публичный клиент нужен пациентам

Внутренний клиент нужен сотрудникам

## Роли

- `PATIENT`
- `REGISTRAR`
- `DOCTOR`
- `CHIEF_DOCTOR`

## Разделение маршрутов

Публичный домен:

- `/`
- `/booking/**`
- `/account/**`

Внутренний домен:

- `/internal/**`
- `/admin/**`

Публичный домен не должен отдавать внутренние маршруты даже после успешного входа

## Режимы SSO

### local

Пользователи живут в Keycloak

Подходит для разработки, демо и маленьких установок

### ad

Keycloak подключается к Active Directory

Windows-клиенты могут получать бесшовный вход через доменную учетную запись и Kerberos

### freeipa

Keycloak подключается к FreeIPA

Linux-клиенты могут получать доменный вход через Kerberos и SSSD

### ldap

Keycloak подключается к LDAP-каталогу

Подходит для инфраструктур без полноценного доменного SSO

### external-oidc

Keycloak доверяет внешнему OIDC-провайдеру

Подходит, если у организации уже есть корпоративный провайдер входа

## Смешанная инфраструктура

Если часть рабочих мест на Windows, а часть на Linux, лучше держать единый центр идентификации

Практичные варианты:

- Active Directory как основной каталог, Linux подключить через SSSD
- FreeIPA как основной каталог, Windows подключать через доверие или отдельный провайдер
- внешний OIDC перед Keycloak, если организация уже использует единый вход

В приложении это не меняет код

Меняется только настройка Keycloak и reverse proxy

## Что приложение не делает

- не авторизует пользователя по IP или имени ПК
- не принимает решение о доменном входе само
- не хранит пароли сотрудников
- не подключается напрямую к AD или FreeIPA

Автовход должен идти через домен или корпоративный провайдер, затем через Keycloak
